# 🖐️ Hand Gesture Recognition – Full Pipeline + MLflow Tracking

**Dataset:** HaGRID (Hand Gesture Recognition Image Dataset) – landmark CSV  
**Goal:** Classify 18 hand gestures from 21 MediaPipe 3-D landmarks per sample  
**Tracking:** MLflow (local `mlruns/` folder)  
**Branch:** `research`

---
### Pipeline
1. Environment Setup  
2. Imports  
3. Data Loading  
4. Data Visualization  
5. Data Preprocessing  
6. MLflow Utility Setup  
7. Model Training + MLflow Logging  
8. Model Evaluation + Comparison  
9. Model Registry  


## ⚙️ Cell 1 — Environment Setup

Run once to install every dependency and initialise the GitHub repo on the `research` branch.

> **After running this cell**, commit with:  
> `git add requirements.txt && git commit -m 'chore: add requirements' && git push origin research`

In [ ]:
# ── Install dependencies ────────────────────────────────────────────────
import subprocess, sys

packages = [
    "mlflow",
    "scikit-learn",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "ipykernel",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])
print("✅ All packages installed")

# ── Save requirements.txt ───────────────────────────────────────────────
reqs = "\n".join([
    "mlflow>=2.13",
    "scikit-learn>=1.4",
    "pandas>=2.0",
    "numpy>=1.26",
    "matplotlib>=3.8",
    "seaborn>=0.13",
    "ipykernel",
])

with open("requirements.txt", "w") as f:
    f.write(reqs)

print("✅ requirements.txt saved")


## 🐙 Cell 2 — GitHub Repo + research Branch

Fill in your details below, then run once.  

> **Commit after this cell:**  
> `git commit --allow-empty -m 'chore: init research branch' && git push origin research`


In [ ]:
import subprocess

# ── FILL THESE IN ──────────────────────────────────────────────────────────
GITHUB_USER  = "YOUR_GITHUB_USERNAME"
GITHUB_EMAIL = "YOUR_GITHUB_EMAIL"
REPO_URL     = "https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPO.git"
# ───────────────────────────────────────────────────────────────────────────

def git(cmd: str):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️  {result.stderr.strip()}")
    else:
        print(f"✅ {result.stdout.strip() or cmd}")

git(f'git config user.name  "{GITHUB_USER}"')
git(f'git config user.email "{GITHUB_EMAIL}"')

# Create / switch to research branch
git("git init")
git("git remote remove origin 2>/dev/null; git remote add origin " + REPO_URL)
git("git checkout -B research")

print("\n🚀 Ready – you are on branch 'research'")


## 📦 Cell 3 — Imports

> **Commit after this cell:**  
> `git add . && git commit -m 'feat: project imports' && git push origin research`


In [ ]:
# Standard
import os, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")
print("✅ Imports OK")


## 📂 Cell 4 — Data Loading

Place `hand_landmarks_data.csv` in the same folder as this notebook  
(or update `DATA_PATH` below).

> **Commit after this cell:**  
> `git add . && git commit -m 'feat: data loading' && git push origin research`


In [ ]:
# ── UPDATE THIS PATH ───────────────────────────────────────────────────────
DATA_PATH = "hand_landmarks_data.csv"
# ──────────────────────────────────────────────────────────────────────────

df = pd.read_csv(DATA_PATH)

print(f"Shape  : {df.shape}")
print(f"Classes: {df['label'].nunique()}")
df.head()


## 📊 Cell 5 — Data Visualization

> **Commit after this cell:**  
> `git add . && git commit -m 'feat: data visualization' && git push origin research`


In [ ]:
# ── MediaPipe landmark connections ────────────────────────────────────────
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
]

def plot_landmark(data):
    labels = sorted(data['label'].unique())
    cols = 6
    rows = (len(labels) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.8, rows * 3))
    axes = axes.flatten()

    for idx, label in enumerate(labels):
        ax = axes[idx]
        sample = data[data['label'] == label].iloc[0]
        xs = [sample[f'x{i}'] for i in range(1, 22)]
        ys = [sample[f'y{i}'] for i in range(1, 22)]

        for s, e in HAND_CONNECTIONS:
            ax.plot([xs[s], xs[e]], [ys[s], ys[e]], color='steelblue', lw=1.5, zorder=1)

        ax.scatter(xs, ys, c='tomato', s=20, zorder=2)
        ax.scatter([xs[0]], [ys[0]], c='gold', s=40, zorder=3)
        ax.set_title(label, fontsize=9, fontweight='bold')
        ax.invert_yaxis(); ax.set_aspect('equal'); ax.axis('off')

    for j in range(len(labels), len(axes)):
        axes[j].axis('off')

    fig.suptitle('One Sample per Gesture Class', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout(); plt.show()

plot_landmark(df)

# ── Class distribution ────────────────────────────────────────────────────
count_df = df['label'].value_counts()
plt.figure(figsize=(10, 5))
sns.barplot(x=count_df.index, y=count_df.values, color='steelblue')
plt.title('HaGRID Classes Distribution')
plt.xlabel('Classes'); plt.ylabel('Count')
plt.xticks(rotation=70); plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()


## 🔍 Cell 6 — Data Investigation

> **Commit after this cell:**  
> `git add . && git commit -m 'feat: data investigation' && git push origin research`


In [ ]:
print("── Null values ──")
print(df.isna().sum().to_string())

print("\n── Duplicates ──")
print(df.duplicated().sum(), "duplicate rows")

print("\n── Descriptive stats ──")
df.describe()


## 🔧 Cell 7 — Normalisation + Label Encoding + Split

Formula:  
`xi_norm = (xi − x_wrist) / x_mid_fingertip`

Split: **60 % train · 20 % val · 20 % test**, `random_state=100`

> **Commit after this cell:**  
> `git add . && git commit -m 'feat: preprocessing and data split' && git push origin research`


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# ── Normalise ─────────────────────────────────────────────────────────────
clean_data = df.copy()
x_wrist, y_wrist = clean_data['x1'], clean_data['y1']
x_mid,   y_mid   = clean_data['x13'], clean_data['y13']

for i in range(1, 22):
    clean_data[f'x{i}'] = (clean_data[f'x{i}'] - x_wrist) / x_mid
    clean_data[f'y{i}'] = (clean_data[f'y{i}'] - y_wrist) / y_mid

# ── Label encode ──────────────────────────────────────────────────────────
le = LabelEncoder()
clean_data['label'] = le.fit_transform(clean_data['label'])

# ── Save processed CSV ────────────────────────────────────────────────────
PROCESSED_PATH = "processed_hand_landmarks_data.csv"
clean_data.to_csv(PROCESSED_PATH, index=False)
print(f"✅ Saved {PROCESSED_PATH}")

# ── Split ─────────────────────────────────────────────────────────────────
data = pd.read_csv(PROCESSED_PATH)
features = data.drop('label', axis=1)
labels   = data['label']

x_train, x_vt, y_train, y_vt = train_test_split(features, labels, test_size=0.4, random_state=100)
x_val,   x_test, y_val, y_test = train_test_split(x_vt,    y_vt,  test_size=0.5, random_state=100)

print(f"Train : {len(x_train):,}  |  Val : {len(x_val):,}  |  Test : {len(x_test):,}")


## 🧪 Cell 8 — MLflow Utility Import + Experiment Setup

Make sure `mlflow_utils.py` is in the **same folder** as this notebook.

> **Commit after this cell:**  
> `git add mlflow_utils.py && git commit -m 'feat: add MLflow utility module' && git push origin research`


In [ ]:
# ── make sure mlflow_utils.py is next to this notebook ────────────────────
import importlib, sys, os

# reload if already imported (helpful during dev)
if "mlflow_utils" in sys.modules:
    importlib.reload(sys.modules["mlflow_utils"])

from mlflow_utils import (
    setup_experiment,
    log_dataset_info,
    log_split_info,
    log_model_params,
    log_metrics,
    log_figure,
    log_model,
    register_model,
    run_experiment,
)
import mlflow

EXP_ID = setup_experiment("hand-gesture-recognition")
print(f"\n✅ MLflow experiment ready  (id={EXP_ID})")
print("   Run: mlflow ui   →  open http://127.0.0.1:5000 in your browser")


## 📐 Cell 9 — Shared Metrics Helper

Defined once, used by every training run.

> **Commit after this cell:**  
> `git add . && git commit -m 'feat: metrics helper' && git push origin research`


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(model, x, y):
    y_pred = model.predict(x)
    return {
        "accuracy"  : accuracy_score(y, y_pred),
        "f1_macro"  : f1_score(y, y_pred, average="macro"),
        "precision" : precision_score(y, y_pred, average="macro"),
        "recall"    : recall_score(y, y_pred, average="macro"),
    }

print("✅ compute_metrics() ready")


## 🔵 Cell 10 — Logistic Regression + MLflow Run

> **Commit after this cell:**  
> `git add mlruns/ && git commit -m 'experiment: logistic-regression run' && git push origin research`


In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(x_train, y_train)

lr_run_id = run_experiment(
    run_name      = "logistic-regression-baseline",
    model         = lr_model,
    model_label   = "LogisticRegression",
    x_train=x_train, y_train=y_train,
    x_val=x_val,     y_val=y_val,
    x_test=x_test,   y_test=y_test,
    df_raw=df,       df_processed=clean_data,
    metrics_fn    = compute_metrics,
)

print(f"\n✅ LR run_id: {lr_run_id}")
print("   Val metrics:", compute_metrics(lr_model, x_val, y_val))


## 🌳 Cell 11 — Decision Tree + MLflow Run

> **Commit after this cell:**  
> `git add mlruns/ && git commit -m 'experiment: decision-tree run' && git push origin research`


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(class_weight='balanced', random_state=42)
dt_model.fit(x_train, y_train)

dt_run_id = run_experiment(
    run_name      = "decision-tree-baseline",
    model         = dt_model,
    model_label   = "DecisionTree",
    x_train=x_train, y_train=y_train,
    x_val=x_val,     y_val=y_val,
    x_test=x_test,   y_test=y_test,
    df_raw=df,       df_processed=clean_data,
    metrics_fn    = compute_metrics,
)

print(f"\n✅ DT run_id: {dt_run_id}")
print("   Val metrics:", compute_metrics(dt_model, x_val, y_val))


## 🔴 Cell 12 — SVM + MLflow Run

> **Commit after this cell:**  
> `git add mlruns/ && git commit -m 'experiment: svm run' && git push origin research`


In [ ]:
from sklearn.svm import SVC

svm_model = SVC(class_weight='balanced', kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_model.fit(x_train, y_train)

svm_run_id = run_experiment(
    run_name      = "svm-rbf-baseline",
    model         = svm_model,
    model_label   = "SVM-RBF",
    x_train=x_train, y_train=y_train,
    x_val=x_val,     y_val=y_val,
    x_test=x_test,   y_test=y_test,
    df_raw=df,       df_processed=clean_data,
    metrics_fn    = compute_metrics,
)

print(f"\n✅ SVM run_id: {svm_run_id}")
print("   Val metrics:", compute_metrics(svm_model, x_val, y_val))


## 🌲 Cell 13 — Random Forest + MLflow Run

> **Commit after this cell:**  
> `git add mlruns/ && git commit -m 'experiment: random-forest run' && git push origin research`


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators  = 100,
    class_weight  = 'balanced',
    random_state  = 42,
    n_jobs        = -1,
)
rf_model.fit(x_train, y_train)

rf_run_id = run_experiment(
    run_name      = "random-forest-n100",
    model         = rf_model,
    model_label   = "RandomForest",
    x_train=x_train, y_train=y_train,
    x_val=x_val,     y_val=y_val,
    x_test=x_test,   y_test=y_test,
    df_raw=df,       df_processed=clean_data,
    metrics_fn    = compute_metrics,
)

print(f"\n✅ RF run_id: {rf_run_id}")
print("   Val metrics:", compute_metrics(rf_model, x_val, y_val))


## 📈 Cell 14 — Model Comparison Chart (logged to MLflow)

> **Commit after this cell:**  
> `git add mlruns/ screenshots/ && git commit -m 'feat: model comparison chart' && git push origin research`


In [ ]:
models_dict = {
    "Logistic\nRegression": lr_model,
    "Decision\nTree":       dt_model,
    "SVM":                   svm_model,
    "Random\nForest":       rf_model,
}

metric_keys  = ["accuracy", "f1_macro", "precision", "recall"]
metric_labels = ["Accuracy", "F1 (macro)", "Precision", "Recall"]
colors       = ['#4C72B0','#55A868','#C44E52','#8172B2']

results = {}
for name, mdl in models_dict.items():
    results[name] = compute_metrics(mdl, x_val, y_val)

# ── Draw ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Model Comparison — Validation Set', fontsize=14, fontweight='bold')

for ax, mkey, mlabel in zip(axes, metric_keys, metric_labels):
    model_names = list(models_dict.keys())
    values      = [results[n][mkey] for n in model_names]
    bars = ax.bar(range(len(model_names)), values, color=colors,
                  width=0.6, edgecolor='white', linewidth=0.8)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom',
                fontsize=7.5, fontweight='bold')

    best_idx = int(np.argmax(values))
    bars[best_idx].set_edgecolor('black')
    bars[best_idx].set_linewidth(2.5)

    short = [n.replace('\n', ' ') for n in model_names]
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(short, fontsize=7.5, rotation=30, ha='right')
    ax.set_title(mlabel, fontsize=10, fontweight='bold')
    ax.set_ylim(max(0, min(values) - 0.05), min(1.0, max(values) + 0.12))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()

# save locally for README + screenshots folder
os.makedirs("screenshots", exist_ok=True)
fig.savefig("screenshots/model_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Log the chart to every run ────────────────────────────────────────────
import mlflow
for run_id in [lr_run_id, dt_run_id, svm_run_id, rf_run_id]:
    with mlflow.start_run(run_id=run_id):
        log_figure(fig, "model_comparison.png", artifact_subdir="plots")

print("\n✅ Comparison chart logged to all runs and saved to screenshots/")


## 🧩 Cell 15 — Confusion Matrix (best model) + MLflow Artifact

> **Commit after this cell:**  
> `git add mlruns/ screenshots/ && git commit -m 'feat: confusion matrix artifact' && git push origin research`


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

y_pred_rf = rf_model.predict(x_test)

fig_cm, ax = plt.subplots(figsize=(12, 10))
cm = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45)
ax.set_title('Random Forest — Confusion Matrix (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig_cm.savefig("screenshots/rf_confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

# log to the RF run
with mlflow.start_run(run_id=rf_run_id):
    log_figure(fig_cm, "rf_confusion_matrix.png", artifact_subdir="plots")

print("✅ Confusion matrix logged")


## 📋 Cell 16 — Results Table

> **Commit after this cell:**  
> `git add . && git commit -m 'feat: results table' && git push origin research`


In [ ]:
print(f"{'Model':<22}  {'Accuracy':>9}  {'F1 macro':>9}  {'Precision':>9}  {'Recall':>9}")
print("─" * 65)

for name, mdl in [("Logistic Regression", lr_model),
                   ("Decision Tree",       dt_model),
                   ("SVM (RBF)",           svm_model),
                   ("Random Forest",       rf_model)]:
    m = compute_metrics(mdl, x_test, y_test)
    marker = " ✅" if name == "Random Forest" else ""
    print(f"{name:<22}  {m['accuracy']:>9.4f}  {m['f1_macro']:>9.4f}  "
          f"{m['precision']:>9.4f}  {m['recall']:>9.4f}{marker}")

print("\n✅ = selected model registered in MLflow Model Registry")


## 🏆 Cell 17 — Register Best Model in MLflow Registry

> **Commit after this cell:**  
> `git add mlruns/ && git commit -m 'feat: register best model in MLflow registry' && git push origin research`


In [ ]:
mv = register_model(
    run_id        = rf_run_id,
    registry_name = "hagrid-gesture-classifier",
)

print(f"\n🏆 Model registered!")
print(f"   Name   : {mv.name}")
print(f"   Version: {mv.version}")
print(f"   Run ID : {mv.run_id}")
print("\nOpen http://127.0.0.1:5000 → 'Models' tab to see the registry entry.")


## 🚀 Cell 18 — Final Git Push

> Run this cell last to make sure **everything** is pushed.


In [ ]:
import subprocess

def git(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = r.stdout.strip() or r.stderr.strip()
    print(f"$ {cmd}\n  {out}\n")

git("git add .")
git('git commit -m "feat: complete hand-gesture-recognition pipeline with MLflow"')
git("git push origin research")

print("🎉 All done! Your 'research' branch is up to date.")
print("   → Open MLflow UI:  mlflow ui")
print("   → Visit: http://127.0.0.1:5000")


## 📝 README Reminder

After running everything, create (or update) `README.md` in your repo root with:

```
## Hand Gesture Recognition — Research Branch

### Model Choice: Random Forest

Random Forest was selected because it achieved the highest accuracy (~93%)
and F1-macro score across all four models, with no hyperparameter tuning —
leaving room for future improvement.

### Model Comparison

| Model               | Accuracy | F1 (macro) | Precision | Recall |
|---------------------|----------|------------|-----------|--------|
| Logistic Regression | ~0.XX    | ~0.XX      | ~0.XX     | ~0.XX  |
| Decision Tree       | ~0.XX    | ~0.XX      | ~0.XX     | ~0.XX  |
| SVM (RBF)           | ~0.XX    | ~0.XX      | ~0.XX     | ~0.XX  |
| **Random Forest** ✅ | **~0.93**| **~0.93**  | **~0.93** | **~0.93**|

*(Fill in exact values from Cell 16 output)*

![Model Comparison](screenshots/model_comparison.png)

### MLflow Registry
Model registered as: `hagrid-gesture-classifier`

### Screenshots
See the `screenshots/` folder for MLflow UI captures.
```
